# **PRÉ-PROCESSAMENTO + REGEX**
**Nome:** Gisela Kassick e Lívia Aragão

## Importações

In [2]:
import re
import pandas as pd
import unicodedata
import spacy
from bs4 import BeautifulSoup

## Configurações

In [14]:
ARQUIVO_ENTRADA = "Eskin_detalhado.xlsx"

ARQUIVO_SAIDA = "dados_preprocessados.xlsx"

# Nome da coluna que contém os abstracts
COLUNA_ABSTRACT = "Abstract"

NGRAM_MIN = 2
NGRAM_MAX = 3

nlp = spacy.load("en_core_web_sm")

componentes_desnecessarios = ["parser", "ner"]

STOPWORDS = set(nlp.Defaults.stop_words)

EXPRESSOES_FIXAS = ["e skin"]

EXPRESSOES_FIXAS_TOKENS = {expressao.lower().replace(" ", "_") for expressao in EXPRESSOES_FIXAS}

PADRAO_FORMULA_QUIMICA = re.compile(
    r"^(?:[A-Z][a-z]?\d*){2,}$"
)

PADROES_BOILERPLATE = [r"copyright\s+(?:©\s*)?\d{4}.*?(?=\.|\n|$)", r"©\s*\d{4}.*?(?=\.|\n|$)", r"all rights reserved.*?(?=\.|\n|$)",

    r"this article is protected by copyright.*?(?=\.|\n|$)", r"this is an open access article.*?(?=\.|\n|$)", r"open access.*?(?=\.|\n|$)",

    r"distributed under the terms of the.*?license.*?(?=\.|\n|$)", r"published under.*?license.*?(?=\.|\n|$)", r"creative commons.*?(?=\.|\n|$)",

    r"https?://doi\.org/\S+", r"doi\s*:\s*10\.\d{4,9}/\S+", r"10\.\d{4,9}/[-._;()/:a-z0-9]+", r"received\s*:?.*?(?=\.|\n|$)",

    r"accepted\s*:?.*?(?=\.|\n|$)", r"submitted\s*:?.*?(?=\.|\n|$)", r"revised\s*:?.*?(?=\.|\n|$)", r"editorial\s*:?.*?(?=\.|\n|$)",

    r"publisher\s*:?.*?(?=\.|\n|$)", r"journal\s*:?.*?(?=\.|\n|$)", r"funding\s*:?.*?(?=\.|\n|$)", r"funded\s+by.*?(?=\.|\n|$)",

    r"this work was supported by.*?(?=\.|\n|$)", r"conflict[s]?\s+of\s+interest.*?(?=\.|\n|$)", r"competing\s+interest[s]?.*?(?=\.|\n|$)",

    r"no\s+conflict[s]?\s+of\s+interest.*?(?=\.|\n|$)", r"data\s+availability.*?(?=\.|\n|$)", r"data\s+are\s+available.*?(?=\.|\n|$)",

    r"data\s+availability\s+statement.*?(?=\.|\n|$)", r"supplementary\s+(?:material|materials|information).*?(?=\.|\n|$)", r"supporting\s+information.*?(?=\.|\n|$)",]

## Normalização unicode + Remoção de boilerplate

In [15]:
def normalizar_unicode(texto):
    """
    Normaliza caracteres Unicode.
    NFKC corrige, entre outras coisas:
    - ligaduras
    - espaços especiais
    - alguns caracteres tipográficos
    """

    texto = unicodedata.normalize("NFKC", texto)

    texto = texto.replace("\xa0", " ")

    return texto


def remover_boilerplate(texto):

    for padrao in PADROES_BOILERPLATE:

        texto = re.sub(padrao, " ", texto, flags=re.IGNORECASE | re.DOTALL)

    return texto

## Limpeza de html/xml e caracteres de controle

In [16]:
def limpar_html_xml(texto):

    if pd.isna(texto) or not texto:

        return ""

    texto = str(texto)

    # Normalização Unicode
    texto = normalizar_unicode(texto)

    # --------------------------------------------------------
    # Remove HTML/XML
    soup = BeautifulSoup(texto, "html.parser")
    texto = soup.get_text(" ")

    # --------------------------------------------------------
    # Remove boilerplate
    texto = remover_boilerplate(texto)

    # --------------------------------------------------------
    # Remove caracteres de controle
    texto = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", "", texto)

    # --------------------------------------------------------
    # Normaliza espaços
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


## Preservar "e-skin"

In [17]:
def preservar_expressoes(texto):

    for expressao in EXPRESSOES_FIXAS:

        partes = expressao.split()
        padrao = r"\s*[-_]?\s*".join(re.escape(parte) for parte in partes)
        texto = re.sub(rf"\b{padrao}\b", "eskin", texto, flags=re.IGNORECASE)

    return texto

## Pré-processamento de um abstract

In [18]:
def preprocessar_abstract(texto):

    # LIMPEZA
    texto_limpo = limpar_html_xml(texto)
    if not texto_limpo:
        return ("","","","",[],[],"","")

    # PRESERVAÇÃO DE EXPRESSÕES COMPOSTAS
    texto_limpo = preservar_expressoes(texto_limpo)

   
    # RETORNO
    return texto_limpo

## Extração por regex

In [19]:
import re
import pandas as pd

def extrair_contexto(texto, palavra, janela=30):
    if pd.isna(texto):
        return None

    texto = str(texto)
    palavras = texto.split()

    # electrode, electrodes, single-electrode, microelectrode, Electrode, ELECTRODE...
    padrao = re.compile(
        rf"(?<!\w){re.escape(palavra)}s?(?!\w)",
        re.IGNORECASE
    )

    contextos = []
    for i, palavra_atual in enumerate(palavras):
        # limpa pontuação das bordas, mas preserva hífen interno
        palavra_limpa = re.sub(r"^[^\w-]+|[^\w-]+$", "", palavra_atual)

        if padrao.search(palavra_limpa):
            inicio = max(0, i - janela)
            fim = min(len(palavras), i + janela + 1)
            contextos.append(" ".join(palavras[inicio:fim]))

    if not contextos:
        return None

    return " ... ".join(contextos)

## Carregar dados

In [20]:
print("CARREGANDO ARQUIVO")

df = pd.read_excel(ARQUIVO_ENTRADA)

print(f"Arquivo: {ARQUIVO_ENTRADA}")
print(f"Linhas: {len(df):,}")
print(f"Colunas: {len(df.columns)}")

print("\nColunas disponíveis:")
print(df.columns.tolist())


# Verificando a coluna de abstract

if COLUNA_ABSTRACT not in df.columns:

    raise ValueError(f"\nA coluna '{COLUNA_ABSTRACT}' não foi encontrada.\nColunas disponíveis:\n{df.columns.tolist()}"    )


CARREGANDO ARQUIVO
Arquivo: Eskin_detalhado.xlsx
Linhas: 5,668
Colunas: 13

Colunas disponíveis:
['Title', 'Year', 'Authors', 'Journal', 'DOI', 'Abstract', 'Citations', 'Type', 'Language', 'Open Access', 'PDF', 'Landing Page', 'OpenAlex ID']


## Aplicar pré-processamento

In [21]:
print("PRÉ-PROCESSANDO ABSTRACTS")

resultados = []

total = len(df)

for i, texto in enumerate(df[COLUNA_ABSTRACT], start=1):

    resultado = preprocessar_abstract(texto)

    resultados.append(resultado)

    # Mostra progresso a cada 1.000 registros
    if i % 1000 == 0 or i == total:

        percentual = (i / total) * 100

        print(f"Processados: {i:,}/{total:,} ({percentual:.1f}%)")



PRÉ-PROCESSANDO ABSTRACTS
Processados: 1,000/5,668 (17.6%)
Processados: 2,000/5,668 (35.3%)
Processados: 3,000/5,668 (52.9%)
Processados: 4,000/5,668 (70.6%)
Processados: 5,000/5,668 (88.2%)
Processados: 5,668/5,668 (100.0%)


In [22]:
# ADICIONA RESULTADOS AO DATAFRAME
df["abstract_limpo"] = resultados

# SALVA
print("SALVANDO RESULTADO")
df.to_excel(ARQUIVO_SAIDA, index=False, engine="openpyxl")


SALVANDO RESULTADO


## Aplicando extração REGEX

In [23]:
df = pd.read_excel(ARQUIVO_SAIDA)

print(f"Total de documentos: {len(df)}")
print(f"Colunas: {list(df.columns)}")


# CONFIGURACAO

PALAVRA = r"electrode"
JANELA = 30


# CRIAR NOVA DF

df_electrode = df[df["abstract_limpo"].fillna("").str.contains(rf"\b{PALAVRA}\b", case=False, regex=True)].copy()


# COLUNA DE CONTEXTO

df_electrode["contexto_electrode"] = df_electrode["abstract_limpo"].apply(lambda x: extrair_contexto(x, PALAVRA, JANELA))


# RESULTADO

print(f"Artigos contendo '{PALAVRA}': " f"{len(df_electrode)}")

print(df_electrode[["abstract_limpo", "contexto_electrode"]].head())

Total de documentos: 5668
Colunas: ['Title', 'Year', 'Authors', 'Journal', 'DOI', 'Abstract', 'Citations', 'Type', 'Language', 'Open Access', 'PDF', 'Landing Page', 'OpenAlex ID', 'abstract_limpo']
Artigos contendo 'electrode': 546
                                       abstract_limpo  \
10  We report human skin based triboelectric nanog...   
11  The performance of robotic and prosthetic hand...   
29  Recently, the quest for new highly stretchable...   
34  Pressure-sensitive electronic skin composed of...   
43  Highly sensitive flexible tactile sensors that...   

                                   contexto_electrode  
10  We constructed a TENG utilizing the contact/se...  
11  rigid core surrounded by a weakly conductive f...  
29  research efforts to expand the promising appli...  
34  Pressure-sensitive electronic skin composed of...  
43  developed by using bionic micropatterned polyd...  


## Salvando em arquivo

In [24]:
df_electrode.to_excel("electrode.xlsx", index=False, engine="openpyxl")

In [27]:
print("NaN no contexto:", df_electrode["contexto_electrode"].isna().sum())

NaN no contexto: 0
